# IRIS-X N10 · Does grounded demonstration feedback improve semantic repair?

這是十份分階段實驗之一。預設停用，儲存為 private inactive QUICK_SAVE。沒有啟動 GPU、訓練、付費執行或競賽 submission。

## 要檢驗的假說
Paired root/seed, same syntax diagnostics and parent source; reveal bounded mismatch evidence versus only no-information repair instruction. Four slots and equal caps, actual tokens reported. This is information-value under matched allocation, not exact realized-compute matching.

- Arms: redacted_demo, true_demo
- 固定 tasks: 8；model calls 上限: 64
- 生成 tokens 上限: 131,072；prompt tokens 上限: 393,216
- 每 call 最多2048新 tokens／6144prompt／8192context；不截斷 demonstrations、query 或修復 source
- N6 host_toolbox semantic admission receipt
- Go/no-go: Report failed-parent-to-demo-fit and final-minus-first selected query exact; no fit-only superiority claim.

## 研究範圍
N4 的61份程式只有45份可 parse、0份通過原 bounded language。這份 suite 先分開 language admission 和 semantic reasoning，再測 representation、feedback、memory 和 search。V2仍是 gas-bounded AST interpreter；只新增受限 while、list methods 等，模型程式從不送入 Python exec/eval。

N5–N12使用原8題已曝光工程題；N14使用原 tuning[8:40] 的32題，這32題已被N3測過，和設計8題不重疊，仍不是新 holdout。136題 confirmation完全不使用。上限相同不代表實際compute相同。N9完整object view在工程題33067df9、N14候選object policy在a096bf4d超過prompt cap；這些exclusion保留在分母，不補題。N13只產生verified synthetic F/G/P/R資料，不訓練或聲稱SFT提升。

## 手動執行
1. 保持目前固定Dataset版本與Internet關閉。N5/N13為CPU-only，不需要CUDA、模型或G4；其他實驗需要既有已验证的G4環境：1×96GB-class RTX PRO6000 Blackwell、48CPUs、host RAM≥170GiB，原CPython3.12/cu128離線包。4×L4 deployment 尚未在本suite實测。
2. 必要時填入實際Dataset mount和PRIOR_RESULTS。前置證據可用本suite輸出的evidence ZIP，或完整解壓資料夾。所需prior runs: N6。不要刪除或修改seal、report、final-inventory等metadata。
3. 願意開始此輪運算時才將 MANUAL_START 改成 True。前置gate失敗會在安裝或載入模型前產生BLOCKED收據，不能跳過gate。
4. 看IRIS-SUITE heartbeat與逐record進度，結尾必須有EXPERIMENT-FINISHED或EXPERIMENT-BLOCKED。下載所列evidence ZIP；未完成、context exclusion、late、unknown work都保留固定分母。
5. 不覆寫／重抽相同run identity；先保存結果。controller上限9600秒；generation120秒為cooperative，並非整個VM硬停止。初始化驗證／interpreter preparation／seal／score／export及清理可能超時。
6. 程式不會停止VM計費。完成後自行停止VM。沒有新增授權GPU/training或自動發布資料。

SOURCE-LOCK: 5483b47b573960c4dba64e2af95efdb1a92e19c2f9bdf652a8b45d9db45e290d


In [ ]:
from pathlib import Path
MANUAL_START = False
RUN_ID = 'N10'
RUNTIME_ROOT = MODEL_ROOT = WHEELS_ROOT = INTERPRETER_ROOT = ADAPTER_ROOT = None
PRIOR_RESULTS = {'N6': None}
BASE = (Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()) / 'iris-nextphase-n10'


In [ ]:
PINS = {'adapter': {'manifest_sha256': '7215085857a3caacbe32302a9b6e77b4ad9524fee2a5a621c39dacb02ee2e9d0', 'private': True, 'reference': 'danielwuu/iris-arc-granite42-run003-adapter-20261009/1'}, 'interpreter': {'manifest_sha256': '20a52944f1ee2abf6f47a2e3beafce44951993439fede39335e660cea143495e', 'private': True, 'reference': 'danielwuu/iris-arc-next-cpython312-offline/2'}, 'model': {'manifest_sha256': 'cd845822cd2c0ca749dd3e6f580f51e68b91c0e5c5886fe239fc3c14f5bb0b99', 'private': True, 'reference': 'danielwuu/iris-arc-granite42-8b-offline/2'}, 'runtime': {'manifest_sha256': 'c2e7913e8ae208deb26e06f37fcd76c79c1a87b6c49ffb3de7b2ee054fee7e67', 'private': True, 'reference': 'danielwuu/iris-arc-next-runtime/8'}, 'wheels': {'manifest_sha256': '2b082a3c36534b5602b8a9f738beec2c01b3d2df11b5438f1c352a84c118b052', 'private': True, 'reference': 'danielwuu/iris-arc-next-cu128-cp312-wheels/1'}}
FILE_HASHES = {'arc_suite.py': 'd2ee8fee3246b8b36f08842e30ce6c8ecc6efea14404d827833c40193d558571', 'arc_suite_policy.py': '78e1f445acf6763b0f8b28c5434d13414e658c8c8a0ec8b0ad61514843271b75', 'arc_bounded_v1.py': '410713680242c0af12f4216181cd0a6318f46788e98e0577b4bfb67b9d2fc911', 'arc_bounded_v2.py': '9f210e97ddea7f98358ce8d3551310fe1411d57db27c21d7d80f705ce5d41ab7', 'arc_n4_program.py': 'fbcf0c933561067b6c9b4fc1e4c44048f02bbd8348ba312d06da37b3f22ebba7', 'arc_program_v2.py': '52bc22ffdf32198ceee209b68f68840869086af7d4372e54ec7d7b0297fe760d', 'arc_typed_ir.py': '8802c4d8e36f2a43acac1abec5a7c2a5b913461cf9f51f7bc7a3eb274ef72ff9', 'arc_curriculum.py': '158263e2e417e02b80a0726c9a436bcd842500b64407d3bc5afa0f595c43238c', 'DESIGN-FROZEN.json': '4a9ade5aec744a2b82f357112c59b2f68bcade25decc3cd794d984837c8dcac9', 'n4-replay-payload.json': '9c508d211ba5cd6d8ab59b2fbeb91456e180467624e77556515869abf7fc042e', 'solver-manifest-reference.json': 'd4f6fbb693d35313f2b9be4eac50e5ac3c8340c61e6b4e77ba71de7a44ea54f8', 'visible-token-audit.json': '90dc7ea352d6d903722f4d645ec87b9c5129d3d6ca425b6a1122e9f5c84c103f', 'SECURITY-RELEASE.json': '22c4fbef103a7ed77681d1df4aa4821108898e4c32043c5749c786a780866a88', 'source-lock.json': '5483b47b573960c4dba64e2af95efdb1a92e19c2f9bdf652a8b45d9db45e290d'}
PACKED_SOURCE_SHA256 = '9e25a3078bb414251e0b5c82d52012449f7b4c73f01a8a5227a314bfbbc70cfb'
UNPACKED_SOURCE_BYTES = 1273787
PACKED_SOURCE_BASE64 = '...'


In [ ]:
import base64, hashlib, importlib.util, importlib.machinery, json, zlib
if type(MANUAL_START) is not bool:
    raise ValueError('Explicit Boolean manual switch required')
if not MANUAL_START:
    print('INACTIVE: '+RUN_ID+' saved. No installation, writes, model load, generation, training, dataset export or submission occurred.')
else:
    packed=base64.b64decode(PACKED_SOURCE_BASE64,validate=True)
    if hashlib.sha256(packed).hexdigest()!=PACKED_SOURCE_SHA256:raise ValueError('Embedded compressed source changed')
    decoder=zlib.decompressobj();raw=decoder.decompress(packed,UNPACKED_SOURCE_BYTES+1)
    if len(raw)!=UNPACKED_SOURCE_BYTES or not decoder.eof or decoder.unused_data or decoder.unconsumed_tail:raise ValueError('Embedded source size/stream differs')
    files=json.loads(raw)
    if set(files)!=set(FILE_HASHES):raise ValueError('Closed embedded source inventory differs')
    for name,value in files.items():
        if Path(name).name!=name or hashlib.sha256(value.encode()).hexdigest()!=FILE_HASHES[name]:raise ValueError('Embedded source hash/path differs')
    def root(role,explicit):
        owner,slug,version=PINS[role]['reference'].split('/')
        choices=[Path(explicit)] if explicit else [Path('/kaggle/input/datasets')/owner/slug,Path('/kaggle/input')/slug]
        found=[p for p in choices if p.is_dir()]
        if len(found)!=1:raise ValueError('Exactly one pinned Dataset mount required: '+role)
        return found[0].absolute()
    required=[] if RUN_ID=='N13' else ['runtime'] if RUN_ID=='N5' else list(PINS)
    roots={role:root(role,globals()[role.upper()+'_ROOT']) for role in required}
    BASE=BASE.absolute()
    if any(p.is_symlink() for p in (BASE,*BASE.parents)) or any(BASE.resolve().is_relative_to(p.resolve()) for p in roots.values()):raise ValueError('Output must be non-symlink and outside every input')
    source=BASE/'source'
    if source.is_symlink():raise ValueError('Symlink source directory forbidden')
    source.mkdir(parents=True,exist_ok=True)
    for name,value in files.items():
        path=source/name
        if path.is_symlink():raise ValueError('Symlink source forbidden')
        if path.exists():
            if hashlib.sha256(path.read_bytes()).hexdigest()!=FILE_HASHES[name]:raise ValueError('Existing source differs')
        else:
            with path.open('xb') as stream:stream.write(value.encode())
    entry=files['arc_suite.py'].encode();name='iris_suite_entry_'+FILE_HASHES['arc_suite.py']
    class VerifiedLoader(importlib.machinery.SourceFileLoader):
        def get_code(self,fullname):return self.source_to_code(entry,str(source/'arc_suite.py'))
    spec=importlib.util.spec_from_file_location(name,source/'arc_suite.py',loader=VerifiedLoader(name,str(source/'arc_suite.py')))
    suite=importlib.util.module_from_spec(spec);spec.loader.exec_module(suite)
    cfg={'output_root':str(BASE/'runs')}
    if RUN_ID!='N13':cfg['runtime_root']=str(roots['runtime'])
    if RUN_ID not in ('N5','N13'):
        cfg.update({role+'_root':str(path) for role,path in roots.items()})
        cfg.update(interpreter_dir=str(BASE/'cpython312'),environment_dir=str(BASE/'environment-cp312-cu128'),pins=PINS)
    result=suite.launch(cfg,RUN_ID,enabled=True,prior_paths={key:value for key,value in PRIOR_RESULTS.items() if value is not None})
    print(json.dumps(result,ensure_ascii=False,indent=2))
